# Derivation: The Policy Gradient and REINFORCE-with-Baseline

In this derivation, we build the policy-gradient framework that Session 3's optional REINFORCE example uses to learn the CES elasticity from a continuous, state-conditioned policy. We motivate the policy-based view, work through the log-derivative trick that produces the policy-gradient theorem, and add a state-dependent baseline that reduces the Monte Carlo estimator's variance without biasing the gradient.

> __Learning Objectives:__
>
> By the end of this notebook, we will be able to:
> * __Why parametrize the policy directly:__ We will see the three structural reasons policy-based RL fits continuous, state-conditioned action selection. We will recognize that stochastic policies build in exploration without an external schedule.
> * __Policy-Gradient Theorem and REINFORCE:__ We will derive the policy-gradient theorem from the log-derivative trick and read off the REINFORCE Monte Carlo estimator. We will see why the environment dynamics never appear in the gradient.
> * __Variance reduction with a baseline:__ We will show that any state-dependent baseline subtracts an unbiased zero from the gradient and use this to derive the advantage-form estimator with a learned linear value baseline.

___


## Why Parametrize the Policy Directly

The bandit and the value-function RL methods (Q-learning, DQN) are *value-based*: they estimate the value of each action and pick the best one. *Policy-based* RL takes the opposite view. We pick a parametrized family of policies $\pi_{\boldsymbol{\theta}}(a \mid s)$, and we move $\boldsymbol{\theta}$ in the direction that increases expected return, the same way we move regression weights in the direction that decreases squared error. The policy itself is the thing we optimize, not the value function.

> __Three structural reasons:__
>
> Three reasons matter for the elasticity-selection problem we have at hand:
>
> * __Continuous actions are natural.__ The CES elasticity $\eta$ lives on $[\eta_{\min}, \eta_{\max}]$. A bandit forces us to pick a discrete grid; a Gaussian policy $\pi_{\boldsymbol{\theta}}(\eta \mid s) = \mathcal{N}(\mu_{\boldsymbol{\theta}}(s), \sigma^{2})$ produces $\eta$ on the full interval and lets gradient information move the mean smoothly toward better values.
> * __State conditioning is automatic.__ Whatever features we put into $s$, the policy mean $\mu_{\boldsymbol{\theta}}(s)$ can use them. With a linear head $\mu_{\boldsymbol{\theta}}(s) = \mathbf{w}_{\mu}^{\top} s + b_{\mu}$, the learned weight vector $\mathbf{w}_{\mu}$ is itself a diagnostic: each entry is the marginal sensitivity of the chosen elasticity to the corresponding feature.
> * __Stochastic policies build in exploration.__ Because $\pi_{\boldsymbol{\theta}}$ is a distribution, sampling from it during training automatically explores. We do not need an $\varepsilon$-greedy schedule; the standard deviation $\sigma$ is itself a learnable parameter that the gradient drives toward a useful exploration scale.

The technical hurdle is that we want to maximize the expected discounted return, defined as:
$$
J(\boldsymbol{\theta}) = \mathbb{E}_{\tau \sim \pi_{\boldsymbol{\theta}}}\!\left[\sum_{t=0}^{T} \gamma^{t} r_{t}\right]
$$
where $\tau = (s_{0}, a_{0}, r_{0}, s_{1}, a_{1}, r_{1}, \ldots)$ is a trajectory whose distribution depends on $\boldsymbol{\theta}$. Naively differentiating $J$ would require differentiating through the environment dynamics, which we do not have a closed form for. The next subsection's *log-derivative trick* (also called the score-function trick) sidesteps this by rewriting the gradient of an expectation as another expectation that the environment never appears in.

___


## The Policy-Gradient Theorem

Write the trajectory probability under policy $\pi_{\boldsymbol{\theta}}$ as $p_{\boldsymbol{\theta}}(\tau)$ and the discounted trajectory return as $R(\tau) = \sum_{t} \gamma^{t} r_{t}$. The expected return takes the integral form:
$$
J(\boldsymbol{\theta}) = \mathbb{E}_{\tau \sim p_{\boldsymbol{\theta}}}\!\left[R(\tau)\right] = \int p_{\boldsymbol{\theta}}(\tau)\, R(\tau)\, d\tau
$$
Taking the gradient and pushing it inside the integral gives:
$$
\nabla_{\boldsymbol{\theta}} J(\boldsymbol{\theta}) = \int \nabla_{\boldsymbol{\theta}} p_{\boldsymbol{\theta}}(\tau)\, R(\tau)\, d\tau
$$
The integrand is not yet an expectation under $p_{\boldsymbol{\theta}}$ because $\nabla_{\boldsymbol{\theta}} p_{\boldsymbol{\theta}}(\tau)$ is not a probability density. The *log-derivative trick* fixes that with the identity:
$$
\nabla_{\boldsymbol{\theta}} p_{\boldsymbol{\theta}}(\tau) = p_{\boldsymbol{\theta}}(\tau)\, \nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\tau)
$$
which converts the gradient back into a probability-weighted form. Substituting yields:
$$
\nabla_{\boldsymbol{\theta}} J(\boldsymbol{\theta}) = \mathbb{E}_{\tau \sim p_{\boldsymbol{\theta}}}\!\left[\nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\tau) \cdot R(\tau)\right]
$$

The trajectory factorizes through the Markov chain $p_{\boldsymbol{\theta}}(\tau) = p(s_{0}) \prod_{t} \pi_{\boldsymbol{\theta}}(a_{t} \mid s_{t})\, p(s_{t+1} \mid s_{t}, a_{t})$, and the trajectory log-density telescopes to:
$$
\nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\tau) = \sum_{t} \nabla_{\boldsymbol{\theta}} \log \pi_{\boldsymbol{\theta}}(a_{t} \mid s_{t})
$$
The initial-state distribution $p(s_{0})$ and the transition kernel $p(s_{t+1} \mid s_{t}, a_{t})$ have no $\boldsymbol{\theta}$-dependence and drop out: *the environment never appears in the gradient*. A causality argument (an action at time $t$ cannot affect rewards at earlier times) then lets us replace the full return $R(\tau)$ with the future-only return $G_{t} = \sum_{k \ge 0} \gamma^{k} r_{t+k}$ in the inner sum without changing the expectation.

The result is the policy-gradient theorem:
$$
\boxed{\;\nabla_{\boldsymbol{\theta}} J(\boldsymbol{\theta}) \;=\; \mathbb{E}_{\tau \sim \pi_{\boldsymbol{\theta}}}\!\left[\sum_{t=0}^{T} \nabla_{\boldsymbol{\theta}} \log \pi_{\boldsymbol{\theta}}(a_{t} \mid s_{t}) \cdot G_{t}\right]\quad\blacksquare\;}
$$

The original results trace back to [Williams (1992)](https://doi.org/10.1007/BF00992696); the modern treatment lives in [Sutton and Barto (2018)](http://incompleteideas.net/book/the-book-2nd.html) chapter 13. Drawing $B$ on-policy trajectories gives the **REINFORCE** Monte Carlo estimator:
$$
\boxed{\;\widehat{\nabla_{\boldsymbol{\theta}} J} \;=\; \frac{1}{B} \sum_{b=1}^{B} \sum_{t=0}^{T} \nabla_{\boldsymbol{\theta}} \log \pi_{\boldsymbol{\theta}}\!\left(a_{t}^{(b)} \mid s_{t}^{(b)}\right) \cdot G_{t}^{(b)}\quad\blacksquare\;}
$$
The intuition is: *increase the log-probability of actions that led to high return, decrease the log-probability of actions that led to low return*. Each $G_{t}$ is a high-variance Monte Carlo estimate of the value of state $s_{t}$, so the estimator is unbiased but converges slowly.

___


## Reducing Variance with a Baseline

REINFORCE is unbiased but high-variance because every trajectory return $G_{t}$ enters the gradient at full magnitude, even when the policy is performing close to its expected level. A *baseline* $b(s)$ is any function of state alone (no action dependence) that we subtract from $G_{t}$ to recenter the rewards before forming the gradient.

The baseline does not bias the estimator. For any fixed state $s$ the expected score is zero:
$$
\mathbb{E}_{a \sim \pi_{\boldsymbol{\theta}}}\!\left[\nabla_{\boldsymbol{\theta}} \log \pi_{\boldsymbol{\theta}}(a \mid s)\right] = \int \pi_{\boldsymbol{\theta}}(a \mid s)\, \frac{\nabla_{\boldsymbol{\theta}} \pi_{\boldsymbol{\theta}}(a \mid s)}{\pi_{\boldsymbol{\theta}}(a \mid s)}\, da = \nabla_{\boldsymbol{\theta}} \int \pi_{\boldsymbol{\theta}}(a \mid s)\, da = \nabla_{\boldsymbol{\theta}}\, 1 = 0
$$
Multiplying by any state-only baseline $b(s)$ preserves the zero, so the gradient with $G_{t}$ replaced by $G_{t} - b(s_{t})$ has the same expectation as the unbaselined version. Variance, however, drops: the optimal state-conditional baseline among scalar functions is the value function $V(s) = \mathbb{E}[G_{t} \mid s_{t} = s]$, which centers the rewards on the policy's currently expected return.

We approximate $V$ with a learned linear head $V_{\boldsymbol{\phi}}(s) = \boldsymbol{\phi}^{\top} s$ trained alongside the policy by Monte Carlo regression on $G_{t}$, and define the **advantage** $A_{t} = G_{t} - V_{\boldsymbol{\phi}}(s_{t})$. The variance-reduced REINFORCE-with-baseline estimator is given by:
$$
\boxed{\;\widehat{\nabla_{\boldsymbol{\theta}} J} \;=\; \frac{1}{B} \sum_{b=1}^{B} \sum_{t=0}^{T} \nabla_{\boldsymbol{\theta}} \log \pi_{\boldsymbol{\theta}}\!\left(a_{t}^{(b)} \mid s_{t}^{(b)}\right) \cdot A_{t}^{(b)}, \qquad A_{t} = G_{t} - V_{\boldsymbol{\phi}}(s_{t})\quad\blacksquare\;}
$$

A linear baseline costs one $n_{s}$-vector update per step and typically halves the gradient variance in practice; richer (neural-network) baselines sit on the same footing without changing the estimator's form.

___


## Summary

The policy-gradient theorem rewrites the gradient of expected return as an expectation of $\nabla_{\boldsymbol{\theta}} \log \pi_{\boldsymbol{\theta}}$ weighted by future return, with the environment dynamics dropping out via the log-derivative trick. REINFORCE is the on-policy Monte Carlo estimator that follows; subtracting a state-dependent baseline reduces its variance without biasing it, and the value function $V_{\boldsymbol{\phi}}(s)$ is the variance-minimizing baseline among scalar state-only functions.

> __Why this matters:__
>
> Session 3's REINFORCE example trains a continuous, state-conditioned elasticity policy that the bandit cannot represent. The policy-gradient theorem is the bridge: it lets us update the policy from realized engine returns without modeling the market dynamics, and the baseline keeps the gradient noise low enough that a few thousand episodes converge to a usable policy.

> __Key Takeaways:__
>
> * __The log-derivative trick eliminates the environment from the gradient:__ Differentiating an expectation under $\pi_{\boldsymbol{\theta}}$ produces an expectation of $\nabla_{\boldsymbol{\theta}} \log \pi_{\boldsymbol{\theta}}$, and the Markov factorization drops the initial-state and transition-kernel terms. We never need a model of $p(s_{t+1} \mid s_{t}, a_{t})$.
> * __REINFORCE is unbiased but noisy:__ Every trajectory return $G_{t}$ enters the gradient at full magnitude, so the estimator works but converges slowly. The variance is the practical bottleneck, not the bias.
> * __State-dependent baselines reduce variance for free:__ Any function of state alone subtracts an expected zero from the gradient, so it cannot bias the estimator. A learned linear value head $V_{\boldsymbol{\phi}}(s) = \boldsymbol{\phi}^{\top} s$ is the cheapest version that captures the level the policy is currently riding.

___
